## 0. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from pathlib import Path

from src.acervo import (
    primeira_baixa_por_processo,
    construir_acervo_historico,
    evolucao_acervo_por_ano,
)

import re

In [ ]:
pip install pyarrow

## 1. Carregar Dados

### Carregar diretórios locais
Os dados processados vêm de `data/processed/` e `data/interim/`, lidos diretamente do repositório local.

In [ ]:
PROCESSED_PATH = Path('data/processed')
INTERIM_PATH = Path('data/interim')
print(f"Processed data path: {PROCESSED_PATH}")
print(f"Interim data path: {INTERIM_PATH}")

### Importando os datasets

In [ ]:
df_concatenados = pd.read_parquet(
    PROCESSED_PATH / 'arquivosConcatenados.parquet',
    engine='pyarrow'
)

In [ ]:
df_andamentos = pd.read_parquet(
    INTERIM_PATH / 'dim_andamentos.parquet',
    engine='pyarrow'
)

## Qual a evolução e perfil do acervo por ano?

Para responder a pergunta "Qual o acervo no dia 31/12/2005?" (acervo anual), nao podemos usar o status atual. Um processo que hoje esta baixado (em 2026) estava plenamente ativo e tramitando em 2005. Por isso, precisamos reconstruir a "linha do tempo" de cada processo, determinando com precisao o seu periodo de vida util.

> A separacao dos dados em tabelas relacionais (Fato e Dimensao) permitiu isolar as variaveis certas sem sobrecarregar a memoria do computador.

### Variáveis Utilizadas na Reconstrução do Acervo

#### Identificador Único: `incidente`
O incidente atua como a chave primária (PK) na tabela fato e chave estrangeira (FK) na tabela de andamentos. É o elo indispensável para rastrear a vida útil de um processo, permitindo cruzar a classe jurídica com todo o seu histórico de movimentações.

#### Marco de Início: `data_protocolo`
Define o nascimento do processo no sistema. Essencial para o cálculo de fluxo de entrada (distribuição) e para determinar o ponto inicial de contagem no estoque anual. Um processo só é contabilizado a partir do momento em que sua data de protocolo é menor ou igual ao limite do ano analisado.

#### Segmentação Jurídica: `classe`
Variável categórica que identifica a natureza da ação (ADI, ADC, ADO, ADPF ou CC). É utilizada para desagregar os dados, permitindo entender o comportamento específico de cada instrumento do controle concentrado de constitucionalidade.

#### Gatilho de Baixa: `and_nome`
Contém a descrição textual dos atos processuais. Através de expressões regulares (Regex), buscamos termos como "Baixa definitiva" ou "Processo findo" para identificar o evento que encerra a tramitação ativa, transformando um dado textual em uma regra de negócio lógica.

#### Marco de Encerramento: `and_data` e `data_baixa`
`and_data` representa a data cronológica de cada movimentação. Após o filtro de baixa, consolidamos a menor data encontrada para cada incidente na variável `data_baixa`. Este é o divisor de águas que determina se o processo deve ser removido do estoque ativo em um determinado ano.

### Descobrir a data exata da primeira "Baixa" de cada processo

Nesta etapa, transformamos o histórico de andamentos (texto) em um marco temporal lógico. A premissa metodológica é identificar o **primeiro evento** que sinaliza a saída do processo do acervo ativo (arquivamento ou baixa definitiva).

**Por que a data mínima?**
Um processo pode ser baixado e reativado por recursos. Ao capturar a primeira data de baixa, garantimos que ele não seja contado indevidamente em anos posteriores ao seu primeiro encerramento, a menos que haja uma reativação explícita (que para este escopo de acervo histórico, tratamos como saída no primeiro marco de encerramento).

In [ ]:
df_concatenados['data_protocolo'] = pd.to_datetime(df_concatenados['data_protocolo'], errors='coerce')

df_primeira_baixa = primeira_baixa_por_processo(df_andamentos).reset_index()
df_primeira_baixa.columns = ['incidente', 'data_baixa']

print(f"Processos únicos com baixa: {df_primeira_baixa['incidente'].nunique()}")

In [ ]:
df_acervo_historico = construir_acervo_historico(
    df_concatenados,
    df_primeira_baixa.set_index('incidente')['data_baixa'],
)

print(f"Total de processos analisados: {len(df_acervo_historico)}")
print(f"Processos já encerrados (histórico): {df_acervo_historico['data_baixa'].notna().sum()}")
print(f"Processos ainda em tramitação: {df_acervo_historico['data_baixa'].isna().sum()}")

#### A Lógica do Snapshot: Como sabemos o que estava ativo no passado?

Para responder com precisão quantos processos existiam no tribunal em, por exemplo, 31 de dezembro de 2005, não podemos olhar para o status atual do processo. Precisamos realizar uma **reconstrução histórica**.

Imagine que cada processo tem uma certidão de nascimento (`data_protocolo`) e pode ou não ter um atestado de encerramento (`data_baixa`). Para um processo ser contado no acervo de um determinado ano, ele precisa atender a dois critérios fundamentais:

1.  **O processo já existia?**
    *   A data em que ele deu entrada no tribunal deve ser anterior ou igual ao último dia do ano que estamos analisando.
    *   *Exemplo:* Se estamos olhando para o ano de 2010, o processo deve ter chegado até 31/12/2010.

2.  **O processo ainda estava tramitando?**
    *   Aqui olhamos para a data de baixa. O processo é considerado **Ativo** se:
        *   Ele ainda não foi baixado (data de baixa está vazia).
        *   **OU** ele foi baixado apenas em um ano futuro.
    *   *Exemplo:* Se o processo foi baixado em 2015, ele ainda era contado como 'Ativo' nos relatórios de 2010 a 2014.

In [ ]:
df_evolucao_acervo = evolucao_acervo_por_ano(df_acervo_historico)

print("Reconstrução histórica concluída com sucesso!")

### Visualização

#### Configuração de ambiente

In [ ]:
import plotly.express as px
import plotly.io as pio

In [ ]:
# Força a renderização correta dentro do ambiente do Google Colab
pio.renderers.default = "colab"

# Dicionário de cores unificado para o projeto (Design Flat/Sofisticado)
cores_classe = {
    "ADI": "#3498db",   # Azul Claro
    "ADC": "#1abc9c",   # Verde Água
    "ADO": "#9b59b6",   # Roxo
    "ADPF": "#e67e22",  # Laranja
}

classes_analise = ["ADI", "ADC", "ADO", "ADPF"]

# Dados agregados por ano (todas as classes somadas)
df_totais_ano = (
    df_evolucao_acervo
    .groupby("ano")[["total_geral", "quantidade_ativos", "quantidade_inativos", "quantidade_baixas", "quantidade_distribuidos"]]
    .sum()
    .reset_index()
)

#### Plotagem Multi-Métrica

Esta célula conterá a inteligência visual do gráfico. Ela aceita qualquer métrica e monta automaticamente o layout emoldurado, os eixos secundários para o Total Geral e a escada histórica de Emendas Regimentais (ER) e ESPIN.

In [ ]:
def plotar_grafico_stf(df_dados, classe_nome, coluna_metrica, label_metrica, titulo_sufixo):
    """
    Gera o gráfico padrão STF.
    Se classe_nome for "TOTAL", plota o acumulado geral em barras simples.
    Se for uma classe (ADI, etc), plota as barras da classe + linha do total geral.
    """
    # 1. Calcular o total geral anual para a métrica
    df_total_geral = df_dados.groupby("ano")[coluna_metrica].sum().reset_index()

    fig = go.Figure()

    # --- CONDIÇÃO SMART: Se o pedido for o TOTAL GERAL ---
    if classe_nome.upper() == "TOTAL":
        fig.set_subplots(specs=[[{"secondary_y": False}]])
        max_y = df_total_geral[coluna_metrica].max()

        # Plota apenas as barras com o total somado de todas as classes
        fig.add_trace(
            go.Bar(
                x=df_total_geral["ano"],
                y=df_total_geral[coluna_metrica],
                marker_color="#3498db",  # Azul padrão para o total
                text=df_total_geral[coluna_metrica],
                textposition="outside",
                cliponaxis=False,
                name=f"Total Geral ({label_metrica})"
            )
        )

    # --- COMPORTAMENTO PADRÃO: Classes Individuais ---
    else:
        fig.set_subplots(specs=[[{"secondary_y": True}]])
        df_filtrado = df_dados[df_dados["classe"] == classe_nome]
        max_y = df_filtrado[coluna_metrica].max()

        # Linha do Total Geral (Eixo Direito - Secundário)
        fig.add_trace(
            go.Scatter(
                x=df_total_geral["ano"], y=df_total_geral[coluna_metrica],
                mode="lines+markers", line=dict(color="#7f7f7f", width=2),
                marker=dict(size=4), name=f"Total Geral ({label_metrica})"
            ),
            secondary_y=True
        )

        # Barras da Classe (Eixo Esquerdo - Primário)
        fig.add_trace(
            go.Bar(
                x=df_filtrado["ano"], y=df_filtrado[coluna_metrica],
                marker_color=cores_classe.get(classe_nome, "#3498db"),
                text=df_filtrado[coluna_metrica], textposition="outside",
                cliponaxis=False, name=f"Classe: {classe_nome}"
            ),
            secondary_y=False
        )

    if max_y == 0 or pd.isna(max_y): max_y = 1

    # 2. Marcos Históricos Estilizados (idênticos para todos)
    fig.add_vrect(x0=2020, x1=2022, fillcolor="green", opacity=0.08, layer="below", line_width=0)
    fig.add_annotation(x=2021, y=max_y * 0.95, text="<b>ESPIN</b>", showarrow=False, xanchor="center", yanchor="top", font=dict(color="green", size=9))

    fig.add_shape(type="line", x0=2016, x1=2016, y0=0, y1=max_y * 1.15, line=dict(color="purple", width=1.2, dash="dash"))
    fig.add_annotation(x=2016, y=max_y * 1.15, text="ER 51", showarrow=False, xanchor="center", yanchor="bottom", font=dict(color="purple", size=9))

    fig.add_shape(type="line", x0=2019, x1=2019, y0=0, y1=max_y * 1.08, line=dict(color="#d9822b", width=1.2, dash="dash"))
    fig.add_annotation(x=2019, y=max_y * 1.08, text="ER 52", showarrow=False, xanchor="center", yanchor="bottom", font=dict(color="#d9822b", size=9))

    fig.add_shape(type="line", x0=2020, x1=2020, y0=0, y1=max_y * 1.00, line=dict(color="red", width=1.2, dash="dash"))
    fig.add_annotation(x=2020, y=max_y * 1.00, text="ER 53", showarrow=False, xanchor="center", yanchor="bottom", font=dict(color="red", size=9))

    # 3. Elementos Fictícios para Legenda
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="purple", width=1.5, dash="dash"), name="ER 51/2016"))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#d9822b", width=1.5, dash="dash"), name="ER 52/2019"))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", width=1.5, dash="dash"), name="ER 53/2020"))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color="green", symbol="square", size=12, opacity=0.2), name="Período da ESPIN"))

    # 4. Ajustes de Layout Condicionais para os Eixos
    titulo_peca = "Total Geral" if classe_nome.upper() == "TOTAL" else f"Classe {classe_nome}"

    fig.update_layout(
        title_text=f"Evolução Anual ({titulo_sufixo}) — {titulo_peca}",
        template="plotly_white", margin=dict(t=120, b=160),
        legend=dict(
            orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5,
            font=dict(size=10, color="#333333"), bgcolor="#fcfcfc", bordercolor="#cccccc", borderwidth=1
        )
    )

    fig.update_xaxes(dtick=1, title_text="Ano de Referência", range=[1987.5, 2025.5])

    if classe_nome.upper() == "TOTAL":
        fig.update_yaxes(title_text=f"Quantidade Total de {label_metrica}")
    else:
        fig.update_yaxes(title_text=f"{label_metrica} da Classe (Barras)", secondary_y=False)
        fig.update_yaxes(title_text=f"Total Geral do Tribunal (Linha)", secondary_y=True)

    fig.show()

### Gráficos

#### Processos ativos

##### Evolução total dos processos ativos anuais

In [ ]:
# Chamando o gráfico macro usando a mesma função!
plotar_grafico_stf(
    df_dados=df_evolucao_acervo,
    classe_nome="TOTAL",
    coluna_metrica="quantidade_ativos",
    label_metrica="Processos Ativos",
    titulo_sufixo="Acervo Ativo Total"
)

###### Análise: Evolução do Acervo Ativo Total

*   **Picos e Estabilização:** O gráfico revela que o STF enfrentou um crescimento quase ininterrupto de processos ativos desde a redemocratização, atingindo um patô crítico entre **2005 e 2012**, onde o acervo superou a marca de 5.000 processos.
*   **Tendência de Queda:** A partir de **2016 (ER 51)**, observa-se o início de uma descida consistente. No entanto, a inclinação da queda torna-se muito mais acentuada a partir de **2019/2020**.
*   **Impacto das ERs:** É notável que a **ER 53/2020** foi o principal catalisador para reduzir o estoque aos níveis atuais (aproximadamente 1.000 processos), representando a maior eficiência na gestão de acervo da história recente do Tribunal.

##### Evolução dos processos ativos anuais por ano por classe

In [ ]:
# --- Gráficos de Acervo Ativo por Classe ---
for classe in classes_analise:
    plotar_grafico_stf(
        df_dados=df_evolucao_acervo,
        classe_nome=classe,
        coluna_metrica="quantidade_ativos",
        label_metrica="Processos Ativos",
        titulo_sufixo="Evolução do Acervo Ativo"
    )

###### Análise: Composição do Acervo por Classe

*   **Hegemonia da ADI:** Conforme a Tabela de Proporções, as ADIs historicamente representaram quase 100% do acervo. Embora ainda sejam o maior volume absoluto, sua proporção caiu de **100% (1988)** para cerca de **70% (2025)**.
*   **Ascensão da ADPF:** A classe ADPF demonstra uma tendência de alta resiliente. Enquanto outras classes caíram drasticamente com as ERs, a ADPF consolidou sua participação, passando de menos de 5% no início dos anos 2000 para **25,7% em 2025**.
*   **Eficiência nas Classes Menores:** ADC e ADO mantêm-se em níveis residuais mínimos, indicando que o Tribunal tem conseguido dar vazão a esses processos quase em tempo real.

#### Baixa


##### Número total de processos que tiveram baixa por ano

In [ ]:
# Gera o gráfico de barras do total geral de baixas por ano (1988 a 2025)
plotar_grafico_stf(
    df_dados=df_evolucao_acervo,
    classe_nome="TOTAL",
    coluna_metrica="quantidade_baixas",
    label_metrica="Processos Baixados",
    titulo_sufixo="Total Geral de Baixas por Ano"
)

###### Análise: Fluxo Anual de Baixas

*   **Explosão de Produtividade:** O gráfico de baixas mostra um salto sem precedentes a partir de **2019**. O recorde histórico de processos encerrados coincide com o período da **ESPIN (2020-2022)**.
*   **Sincronia Regimental:** O aumento das saídas valida a eficácia da **ER 53**, que permitiu o julgamento acelerado em ambiente virtual.
*   **Contraste Histórico:** Antes de 2015, o tribunal mantinha uma média de baixas baixa e estável, o que explica por que o acervo ativo acumulava tanto naqueles anos.

##### Números de processos que tiveram baixa por ano por classe

In [ ]:
# --- Gráficos de Baixas Processuais por Classe ---
for classe in classes_analise:
    plotar_grafico_stf(
        df_dados=df_evolucao_acervo,
        classe_nome=classe,
        coluna_metrica="quantidade_baixas",
        label_metrica="Processos Baixados",
        titulo_sufixo="Fluxo Anual de Baixas"
    )

###### Análise: Fluxo de Baixas por Classe

*   **Picos de Encerramento:** A classe **ADI** registra os maiores picos absolutos de baixas, especialmente entre 2019 e 2022, refletindo o esforço concentrado do Tribunal em limpar o estoque histórico.
*   **Eficiência na ADPF:** Nota-se que as baixas de **ADPFs** acompanham o ritmo de sua distribuição; embora entrem muitos processos desta classe, a saída também é acelerada, impedindo a formação de gargalos como os vistos nas ADIs no passado.
*   **Impacto Normativo:** A partir da **ER 53**, todas as classes apresentaram um aumento na velocidade de encerramento, validando o modelo de julgamento virtual para diferentes naturezas de ação.

#### Distribuições

##### Número total processos distribuidos por ano

In [ ]:
# Gráfico macro do fluxo de entrada anual do tribunal
plotar_grafico_stf(
    df_dados=df_evolucao_acervo,
    classe_nome="TOTAL",
    coluna_metrica="quantidade_distribuidos",
    label_metrica="Processos Distribuídos",
    titulo_sufixo="Volume Total de Distribuição Anual"
)

###### Análise: Volume de Distribuição

*   **Picos de Entrada:** O tribunal sofreu grandes ondas de novas ações em **2005** e novamente em **2020**. O pico de 2020 está diretamente ligado ao aumento de litígios constitucionais durante a pandemia.
*   **O Segredo da Queda do Acervo:** A análise comparativa mostra que, embora a entrada de processos (Distribuição) tenha continuado alta, o volume de **Baixas** foi significativamente superior. É esse diferencial positivo (sair mais do que entra) que permitiu a redução histórica do estoque ativo vista nos gráficos anteriores.

##### Numero de processos distribuidos por ano por classe

In [ ]:
# --- Gráficos de Distribuição de Novos Processos ---
for classe in classes_analise:
    plotar_grafico_stf(
        df_dados=df_evolucao_acervo,
        classe_nome=classe,
        coluna_metrica="quantidade_distribuidos",
        label_metrica="Processos Distribuídos",
        titulo_sufixo="Novos Processos Distribuídos"
    )

###### Análise: Distribuição de Novos Processos por Classe

*   **Estabilidade da ADI:** A entrada de novas **ADIs** mantém um patamar histórico relativamente constante, com oscilações pontuais, mas sem o crescimento explosivo visto em décadas anteriores.
*   **O Salto da ADPF:** O gráfico torna evidente o protagonismo recente da **ADPF**. A partir de 2018-2019, houve um salto significativo nas distribuições, com um pico notável em 2020 relacionado à judicialização de políticas públicas durante a pandemia.
*   **Classes Residuais:** As distribuições de **ADC** e **ADO** permanecem baixas e estáveis, reforçando que são instrumentos utilizados de forma muito mais específica e criteriosa pelos legitimados.

#### Tabela de Proporção do Acervo por Classe
Esta tabela apresenta a distribuição absoluta e percentual de cada classe processual em relação ao acervo total de cada ano.

In [ ]:
# 1. Criar a base pivotada (Anos x Classes)
df_tabela_detalhada = df_evolucao_acervo.pivot(
    index='ano',
    columns='classe',
    values='quantidade_ativos'
).fillna(0).astype(int)

# 2. Calcular o Total Geral por ano
df_tabela_detalhada['TOTAL_GERAL'] = df_tabela_detalhada.sum(axis=1)

# 3. Calcular as proporções (percentuais) para cada classe
for classe in ['ADC', 'ADI', 'ADO', 'ADPF']:
    df_tabela_detalhada[f'%_{classe}'] = (
        (df_tabela_detalhada[classe] / df_tabela_detalhada['TOTAL_GERAL']) * 100
    ).round(2)

# Reordenar colunas para facilitar a leitura (Classe, depois seu respectivo %)
colunas_ordenadas = ['TOTAL_GERAL']
for classe in ['ADC', 'ADI', 'ADO', 'ADPF']:
    colunas_ordenadas.append(classe)
    colunas_ordenadas.append(f'%_{classe}')

display(df_tabela_detalhada[colunas_ordenadas])

classe,TOTAL_GERAL,ADC,%_ADC,ADI,%_ADI,ADO,%_ADO,ADPF,%_ADPF
ano,,,,,,,,,
1988,11,0,0.00,11,100.00,0,0.00,0,0.00
1989,160,0,0.00,160,100.00,0,0.00,0,0.00
1990,397,0,0.00,397,100.00,0,0.00,0,0.00
1991,611,0,0.00,611,100.00,0,0.00,0,0.00
1992,703,0,0.00,703,100.00,0,0.00,0,0.00
1993,774,1,0.13,773,99.87,0,0.00,0,0.00
1994,883,1,0.11,882,99.89,0,0.00,0,0.00
1995,982,0,0.00,982,100.00,0,0.00,0,0.00
1996,1046,0,0.00,1046,100.00,0,0.00,0,0.00


### Conclusão Geral: A Transformação do Controle Concentrado (1988 - 2025)

A reconstrução histórica do acervo e as visualizações métricas permitem concluir que o Supremo Tribunal Federal atravessa seu período de maior transformação operacional desde a Constituição de 1988.

#### 1. Eficiência Inédita na Gestão do Estoque
Os dados confirmam que o acervo ativo atingiu seu ápice entre **2005 e 2012**. A partir de **2019**, com a introdução das **ER 52 e 53**, o tribunal rompeu a barreira da acumulação histórica. O segredo da queda drástica de ~5.000 para ~1.000 processos ativos não foi a diminuição de trabalho (as entradas continuam altas), mas sim uma **explosão de produtividade nas saídas (Baixas)**, superando consistentemente o volume de novas distribuições.

#### 2. Mudança de Paradigma: De ADI para ADPF
O perfil jurídico do tribunal mudou. Enquanto as **ADIs** estabilizaram seu crescimento e tiveram seu estoque reduzido, as **ADPFs** assumiram um papel de protagonismo absoluto na última década. A Tabela de Proporções é clara: a ADPF saltou de uma participação marginal para representar **mais de 25% do acervo ativo em 2025**, consolidando-se como o instrumento preferencial para crises institucionais e direitos fundamentais.

#### 3. Legado da ESPIN e Plenário Virtual
O período da **ESPIN (2020-2022)**, longe de ser um período de paralisação, funcionou como um acelerador tecnológico. O uso intensivo do **Plenário Virtual** permitiu que o tribunal mantivesse o ritmo de baixas em níveis recordes mesmo durante a pandemia. Esse modelo provou ser resiliente e é hoje o principal responsável por evitar o retorno ao congestionamento processual do passado.

#### 4. Perspectivas Futuras
Com o acervo histórico de ADIs sob controle, o desafio do STF para os próximos anos desloca-se da **quantidade bruta** para a **complexidade qualitativa**. O tribunal entra em 2025 com um acervo mais "limpo" e atualizado, permitindo que a pauta se concentre em temas de alto impacto social e constitucional com maior celeridade.

### Salvando o dataset da evolução do acervo

In [ ]:
ACERVO_OUT_PATH = PROCESSED_PATH / 'acervo'
ACERVO_OUT_PATH.mkdir(parents=True, exist_ok=True)

df_evolucao_acervo.to_parquet(
    ACERVO_OUT_PATH / 'evolucao_acervo.parquet',
    engine='pyarrow',
    index=False
)

print(f"Arquivo salvo com sucesso em: {ACERVO_OUT_PATH / 'evolucao_acervo.parquet'}")

In [ ]:
df_evolucao_acervo

,ano,classe,total_geral,quantidade_ativos,quantidade_inativos,quantidade_baixas,quantidade_distribuidos
0,1988,ADI,11,11,0,0,11
1,1988,ADC,0,0,0,0,0
2,1988,ADO,0,0,0,0,0
3,1988,ADPF,0,0,0,0,0
4,1988,CC,0,0,0,0,0
...,...,...,...,...,...,...,...
185,2025,ADI,7849,726,7123,304,143
186,2025,ADC,100,22,78,3,7
187,2025,ADO,92,18,74,5,3
188,2025,ADPF,1285,265,1020,76,98
